# 02 — Table Order Items

Nettoyage de la table `order_items` : une ligne par article commandé, grain du DataFrame final.

*Extrait de `ETL-goupe.ipynb` (commit `72541c3`).*

On commence par importer les librairies nécessaires et charger les fichiers CSV dans un dictionnaire de DataFrames. Ensuite, nous créons une copie du dictionnaire original pour conserver les données brutes.

In [1]:
# Imports des bibliothèques

import glob

import pandas as pd

# afficher toutes les colonnes
pd.set_option("display.max_columns", 50)

all_files = sorted(glob.glob("*.csv"))

df_dic = {}
df_origin = {}
for file in all_files:
    nom = file.replace("datasets/Olist/", "").replace("olist_", "").replace("_dataset", "").replace(".csv", "")
    df_dic[nom] = pd.read_csv(file)
    df_origin[nom] = pd.read_csv(file)


def resume(df):
    """Renvoie un tableau de synthèse: une ligne par colonne  de df."""
    return pd.DataFrame(
        {
            "type": df.dtypes,
            "nb_uniques": df.nunique(),
            "maximum": df.max(),
            "minimum": df.min(),
            "nb manquants": df.isnull().sum(),
            "pct_manquants": (df.isnull().mean() * 100).round(2),
        }
    )

### Transformation de la table Order Items

In [2]:
items = df_dic["order_items"]

display(resume(items))

print("Dimensions :", items.shape)
print("Doublons :", items.duplicated().sum())

display(items.isna().sum())
display(items.describe())
display(items.dtypes)

,type,nb_uniques,maximum,minimum,nb manquants,pct_manquants
order_id,str,98666,fffe41c64501cc87c801fd61db3f6244,00010242fe8c5a6d1ba2dd792cb16214,0,0.0
order_item_id,int64,21,21,1,0,0.0
product_id,str,32951,fffe9eeff12fcbd74a2f2b007dde0c58,00066f42aeeb9f3007548bb9d3f33c38,0,0.0
seller_id,str,3095,ffff564a4f9085cd26170f4732393726,0015a82c2db000af6aaaf3ae2ecb0532,0,0.0
shipping_limit_date,str,93318,2020-04-09 22:35:08,2016-09-19 00:15:34,0,0.0
price,float64,5968,6735.0,0.85,0,0.0
freight_value,float64,6999,409.68,0.0,0,0.0


Dimensions : (112650, 7)
Doublons : 0


order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64

,order_item_id,price,freight_value
count,112650.000000,112650.000000,112650.000000
mean,1.197834,120.653739,19.990320
std,0.705124,183.633928,15.806405
min,1.000000,0.850000,0.000000
25%,1.000000,39.900000,13.080000
50%,1.000000,74.990000,16.260000
75%,1.000000,134.900000,21.150000
max,21.000000,6735.000000,409.680000


order_id                   str
order_item_id            int64
product_id                 str
seller_id                  str
shipping_limit_date        str
price                  float64
freight_value          float64
dtype: object

### Conversion de la colonne shipping_limit_date en datetime


In [3]:
items["shipping_limit_date"] = pd.to_datetime(
    items["shipping_limit_date"]
)

display(resume(items))

,type,nb_uniques,maximum,minimum,nb manquants,pct_manquants
order_id,str,98666,fffe41c64501cc87c801fd61db3f6244,00010242fe8c5a6d1ba2dd792cb16214,0,0.0
order_item_id,int64,21,21,1,0,0.0
product_id,str,32951,fffe9eeff12fcbd74a2f2b007dde0c58,00066f42aeeb9f3007548bb9d3f33c38,0,0.0
seller_id,str,3095,ffff564a4f9085cd26170f4732393726,0015a82c2db000af6aaaf3ae2ecb0532,0,0.0
shipping_limit_date,datetime64[us],93318,2020-04-09 22:35:08,2016-09-19 00:15:34,0,0.0
price,float64,5968,6735.0,0.85,0,0.0
freight_value,float64,6999,409.68,0.0,0,0.0


## Les livraisons gratuites ne doivent pas être considérées automatiquement comme des erreurs.

In [4]:
print("Prix négatifs :", (items["price"] < 0).sum())

print("Prix nuls :", (items["price"] == 0).sum())

print("Frais négatifs :", (items["freight_value"] < 0).sum())

print("Livraisons gratuites :", (items["freight_value"] == 0).sum())

Prix négatifs : 0
Prix nuls : 0
Frais négatifs : 0
Livraisons gratuites : 383


### Correction des anomalies confirmées


In [5]:
items = items.drop_duplicates().copy()

display(items[
    (items["price"] <= 0)
    | (items["freight_value"] < 0)
])
df_dic["order_items"] = items

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value


### Comparaison des données avant et après transformation


In [6]:
for nom in ["order_items"]:

    avant = df_origin[nom]
    apres = df_dic[nom]

    comparaison = pd.DataFrame({
        "Avant": [
            len(avant),
            avant.shape[1],
            avant.duplicated().sum(),
            avant.isna().sum().sum()
        ],
        "Après": [
            len(apres),
            apres.shape[1],
            apres.duplicated().sum(),
            apres.isna().sum().sum()
        ]
    }, index=[
        "Nombre de lignes",
        "Nombre de colonnes",
        "Doublons",
        "Valeurs manquantes"
    ])

    print(nom)
    display(comparaison)

order_items


,Avant,Après
Nombre de lignes,112650,112650
Nombre de colonnes,7,7
Doublons,0,0
Valeurs manquantes,0,0
